<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c35-header.png" alt="Nextia Learning · Forecasting and Time Series: Predict What Comes Next" width="100%">

# Naive forecasts

**[Naive forecasts](https://learning.nextia-ai.com/courses/forecasting/m03/naive-forecasts/)** · Forecasting and Time Series: Predict What Comes Next · Module 3, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** make the four simplest forecasts (the last value, the same day last week, a moving average and drift), and to see why every later method must beat them.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | Trend and seasons, from [Trend, season and noise](https://learning.nextia-ai.com/courses/forecasting/m02/trend-season-and-noise/). Why a random split is wrong for a series, from [Yesterday predicts today](https://learning.nextia-ai.com/courses/forecasting/m02/yesterday-predicts-today/). |
| **Tested** | Python 3.12 with pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, hierarchicalforecast 1.5.3, LightGBM 4.7.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m03/naive-forecasts/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C35/M03-L01-naive-forecasts/naive-forecasts-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 2.3.3, statsforecast 2.1.1, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==2.3.3 statsforecast==2.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import statsforecast; print("statsforecast", statsforecast.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield data of this course (about 2 MB) into a folder `larkfield-forecasting/data`, and the course's backtest helper `backtest.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 403758 rows of daily sales (200 products, 3 regions), from 2024-04-01 to 2026-03-29.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-forecasting` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: we made it for this course. It has no real people, products or prices.

In [ ]:
# Setup: download the course data into larkfield-forecasting/data/ and check it.
import gzip, hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C35/"
FILES = {
    "calendar.csv": "a5d1d5a3b640a983de72a30fca0e627a4ef23e797b1d80619e9ffcaadf118844",
    "demand.csv.gz": "8b39303985cbdb9bb1f3c705cd927ed6aaeb6f9665bd56ad4c3f8cb2b8578c99",
    "orders.csv": "bd8059fd73fe294daa157f8a6b6ae3f58319fa644eaa7ff0d31122b02cf05e30",
    "price_changes.csv": "889b258be4e933b87065d81d0771d6cd9b630ee90223b0a6ab074129c42153cb",
    "products.csv": "198b0a3de6eea8128af68f3cbc3a5c6acd3a64ac854fff18d37660857be8cfbf",
    "promotions.csv": "ca147959faa4c36c757742de7121976ffe79d59b53049dbbc8337a7547541f4a",
    "roster_hourly.csv": "6c2e0e96881b36ded52ed19458515c71f025288ddaebd589b50c82a8ad7fa69b",
    "tickets_daily.csv": "36f7067db060fdd2bd084c408c0622370a012773849f243020336f1fc678b31d",
    "tickets_hourly.csv.gz": "43f7fbb3e360a8e4a0cb4e101ea6a6242a76f0c37204d76ee9413ea3adbedd92",
    "weather.csv": "43a199a8b235940e291195bfd026901b62a10189b29f3a3ad9527b5382544276",
    "backtest.py": "764ecd38c6e96632be6f0e139e05eadd7b07d14a4ece73d881fc7559c2fa203a",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-forecasting").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and sha256(path) == expected:
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-forecasting and run again.")

for name, expected in FILES.items():
    if name == "backtest.py":
        fetch(name, LABS + name, PROJECT / name, expected)
    else:
        fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
with gzip.open("data/demand.csv.gz", "rt") as f:
    header = f.readline()
    rows = [line.split(",", 3)[:3] for line in f]
print(f"Ready: {len(rows)} rows of daily sales ({len({r[2] for r in rows})} products, {len({r[1] for r in rows})} regions), "
      f"from {min(r[0] for r in rows)} to {max(r[0] for r in rows)}.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course's backtest helper as `bt`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the backtest helper, and checks that do not show the answer.
import hashlib, json, logging, os, warnings
os.environ["NIXTLA_ID_AS_COL"] = "1"
import numpy as np
import pandas as pd
import backtest as bt
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")
logging.getLogger("lightning").setLevel(logging.ERROR)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def load_sales(products=None):
    """Daily sales per product, the three regions added up, in the long format of the backtest:
    unique_id (the product), ds (the day), y (units sold), price, on_promotion, stock_out.
    The missing day 2025-10-19 gets the mean of the same weekday one week before and after."""
    d = pd.read_csv("data/demand.csv.gz", parse_dates=["date"])
    if products is not None:
        d = d[d["product_id"].isin(products)]
    g = d.groupby(["product_id", "date"], as_index=False).agg(
        y=("sales", "sum"), price=("price", "mean"), on_promotion=("on_promotion", "max"), stock_out=("stock_out", "max"))
    out = []
    for uid, s in g.rename(columns={"product_id": "unique_id", "date": "ds"}).groupby("unique_id"):
        s = s.set_index("ds").reindex(pd.date_range(s["ds"].min(), s["ds"].max(), freq="D"))
        gap = s["y"].isna()
        before, after = s["y"].shift(7), s["y"].shift(-7)
        s.loc[gap, "y"] = ((before + after) / 2).fillna(before).fillna(after)[gap].round()
        s = s.ffill()
        s["unique_id"] = uid
        out.append(s.rename_axis("ds").reset_index())
    return pd.concat(out, ignore_index=True)[["unique_id", "ds", "y", "price", "on_promotion", "stock_out"]]

# The course backtest (the same in every module): 13 origins, 28 days apart, the last one 2026-03-15.
ORIGINS = bt.origins("2026-03-15", n=13, every="28D")
BACKTEST_PRODUCTS = ['P001', 'P002', 'P005', 'P006', 'P009', 'P013', 'P018', 'P024', 'P025', 'P029', 'P032', 'P039', 'P045', 'P057', 'P068', 'P075', 'P077', 'P078', 'P080', 'P081', 'P087', 'P088', 'P093', 'P095', 'P108', 'P110', 'P121', 'P130', 'P133', 'P136', 'P137', 'P141', 'P145', 'P147', 'P148', 'P156', 'P158', 'P159', 'P166', 'P169', 'P172', 'P174', 'P176', 'P183', 'P184', 'P186', 'P191', 'P192', 'P194', 'P199']
PRICES = pd.read_csv("data/products.csv").set_index("product_id")["base_price"].to_dict()

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Four forecasts by hand

A product sold these units in the last two weeks, Monday to Sunday: `10, 8, 9, 11, 14, 20, 18` and `12, 9, 10, 12, 15, 22, 19`. What will it sell next Monday, Tuesday and Wednesday?

- **Naive:** the last value, every day: 19, 19, 19.
- **Seasonal naive:** the same weekday last week: 12, 9, 10.
- **Moving average (7 days):** the mean of the last 7 days, every day.
- **Drift:** the last value plus the average change per day over the whole history.

> **Predict.** Which forecast looks right for a Monday? Then run the cell: it uses the four functions of `backtest.py`.

In [ ]:
history = pd.DataFrame({"unique_id": "shop", "ds": pd.date_range("2025-06-02", periods=14, freq="D"),
                        "y": [10, 8, 9, 11, 14, 20, 18, 12, 9, 10, 12, 15, 22, 19]})
future = pd.DataFrame({"unique_id": "shop", "ds": pd.date_range("2025-06-16", periods=3, freq="D")})
for name, f in [("naive", bt.naive), ("seasonal naive", bt.seasonal_naive(7)),
                ("moving average 7", bt.moving_average(7)), ("drift", bt.drift)]:
    print(f"{name:17}", f(history, 3, future)["yhat"].round(2).tolist())

> **Check.** You should see `naive [19, 19, 19]`, `seasonal naive [12, 9, 10]`, `moving average 7 [14.14, 14.14, 14.14]` and `drift [19.69, 20.38, 21.08]`.

The moving average is (12 + 9 + 10 + 12 + 15 + 22 + 19) ÷ 7 = 99 ÷ 7 = 14.14. The drift is (19 − 10) ÷ 13 = 0.69 a day. Only the seasonal naive knows that Monday is a quiet day. The others forget the weekly season.

## 2. The parasol, before the heat wave

Kenji orders on Mondays. The next cell takes the parasol's history up to Sunday 22 June 2025, makes the four forecasts for the next 14 days, and draws them with what was really sold.

> **Predict.** The heat wave starts on 30 June. Which forecast will be closest in the second week? Then run the cell.

In [ ]:
import matplotlib.pyplot as plt

parasol = load_sales(["P025"])
origin = pd.Timestamp("2025-06-22")
train = parasol[parasol["ds"] <= origin]
real = parasol[(parasol["ds"] > origin) & (parasol["ds"] <= origin + pd.Timedelta(days=14))]
future = real.drop(columns="y")
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(train["ds"].tail(42), train["y"].tail(42), color="grey", label="history")
ax.plot(real["ds"], real["y"], "k.-", label="real sales")
for name, f in [("naive", bt.naive), ("seasonal naive", bt.seasonal_naive(7)),
                ("moving average 28", bt.moving_average(28)), ("drift", bt.drift)]:
    fc = f(train, 14, future)
    ax.plot(fc["ds"], fc["yhat"], label=name)
    print(f"{name:17} 14-day total {fc['yhat'].sum():6.1f}   real {real['y'].sum():.0f}")
ax.set_xlabel("Day")
ax.set_ylabel("Parasols sold")
ax.legend()
plt.show()

> **Check.** You should see the real total 197 and these forecast totals: naive 182.0, seasonal naive 134.0, moving average 133.0, drift 185.1.

All four are too low: none of them can know the heat wave. The last value and drift come closest, but only by luck: the last day of the history was a busy Sunday, and they repeat it every day. A simple forecast is not a good forecast; it is a **baseline**, the score that a better method must beat.

## 3. The four baselines in the course backtest

The next cell runs the four methods on the 50 products, at the 13 cut-off days of the course, for 14 days each, and prints their rows of the backtest table. Module 3, lesson 3 explains the columns. For now, read **WAPE %**: the absolute misses added up, in percent of the units sold. Lower is better.

> **Predict.** Which one wins on WAPE: the last value, the same weekday last week, the mean of 28 days or drift?

In [ ]:
sales = load_sales(BACKTEST_PRODUCTS)
rows = []
forecasts = {}
for name, f in [("naive", bt.naive), ("seasonal naive", bt.seasonal_naive(7)),
                ("moving average (28 days)", bt.moving_average(28)), ("drift", bt.drift)]:
    forecasts[name] = bt.backtest(sales, f, h=14, origins=ORIGINS)
    rows.append(bt.score(forecasts[name], sales, name, season=7, cost=bt.stock_cost(PRICES)))
table = bt.show(rows)

> **Check.** You should see four rows, best first: moving average (28 days) with WAPE % 43.5, seasonal naive 51.3, naive 53.2 and drift 53.8.

A surprise: the 28-day mean beats "the same weekday last week" on most products. One week of a noisy product is a poor guide; 28 days average the noise away. But the seasonal naive has the **lower cost** of the stock order: its misses are larger day by day, but they cancel more over the 14 days of an order. Which method is "best" depends on the decision.

## 4. Your turn: a shorter window

> **Your turn.** Backtest the moving average over **7 days** (`bt.moving_average(7)`) on `sales`, with `h=14` and `origins=ORIGINS`, and score it with `bt.score(..., season=7)`. Put its **WAPE %** in `wape_ma7` (a number from the score's dictionary). Then run the check cell.

In [ ]:
wape_ma7 = ...  # the WAPE % of the 7-day moving average

In [ ]:
expect('wape_ma7', wape_ma7, 42.1)

Here the 7-day window wins a little (42.1% against 43.5%): it follows the spring and the autumn faster. It also follows more noise, and on another set of products it can lose. The window is a choice that you test in the backtest, not one that you guess.

## 5. Failure case: a planned promotion

Product P141 (a duvet) had a 25% promotion from 8 to 14 September 2025. The promotion was planned: it is in `promotions.csv`. The next cell makes the seasonal naive forecast on Sunday 7 September.

In [ ]:
duvet = load_sales(["P141"])
origin = pd.Timestamp("2025-09-07")
train = duvet[duvet["ds"] <= origin]
real = duvet[(duvet["ds"] > origin) & (duvet["ds"] <= origin + pd.Timedelta(days=7))]
fc = bt.seasonal_naive(7)(train, 7, real.drop(columns="y"))
print("forecast", int(fc["yhat"].sum()), "| real", int(real["y"].sum()), "| promotion days:", int(real["on_promotion"].sum()))

> **Check.** You should see `forecast 80 | real 183 | promotion days: 7`.

The forecast misses about half of the week, although everybody at Larkfield knew about the promotion. A naive forecast uses only the past values of the series. **Known future events need a method that can use them**: Module 2, lesson 3 puts them in a calendar, and Module 5 gives them to a model.

## Next

You have four baselines and their rows in the backtest table. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m03/naive-forecasts/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Test like the future: backtesting](https://learning.nextia-ai.com/courses/forecasting/m03/test-like-the-future-backtesting/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/forecasting/m03/naive-forecasts/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.